# DNS Resolution & C2 Beacon Detection Graph

> **Platform:** Microsoft Sentinel Custom Graph | **Domain:** Network Security
> **Data Sources:** DeviceNetworkEvents, DeviceInfo, ThreatIntelIndicators
> **Nodes:** 5 (Device, Domain, ParentDomain, ResolvedIP, ThreatIndicator)
> **Edges:** 5 (Queried, SubdomainOf, ResolvesTo, MatchesTI [IP], MatchesTI [Domain])

Models DNS resolution patterns with pre-computed beaconing metrics (interval regularity, coefficient of variation, active hours) to detect C2 beacon infrastructure through endpoint-domain-IP threat mapping.

## Environment & Configuration

Configure the environment and verify SDK availability. The notebook requires:
- **`MicrosoftSentinelProvider`** -- reads tables from the Sentinel Data Lake
- **`GraphSpecBuilder`** -- fluent API to define nodes, edges, and build the graph
- **`Graph`** -- builds, publishes, and queries the graph instance

In [ ]:
# 📅 OPTIONAL: Adjust the lookback window (default: 7 days). Use 1 for IR, 14 for hunting, 30+ for historical.
TIME_WINDOW_DAYS = 1

import logging

# Set the logging level for the entire package
logging.getLogger('sentinel_graph').setLevel(logging.INFO)
print(f" Logging level set to: {logging.getLevelName(logging.getLogger('sentinel_graph').level)} and above")


## Data Ingestion

Read source tables from the Sentinel Data Lake, apply time filters, and persist to Spark memory.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from sentinel_graph import GraphSpecBuilder, Graph
from pyspark.sql.functions import (
 col, lit, expr, count, first, last, concat_ws,
 split, size, concat, array_join, slice as spark_slice,
 regexp_extract, min as spark_min, max as spark_max,
 from_json, get_json_object
)
from pyspark.sql.types import StringType, ArrayType

sentinel_provider = MicrosoftSentinelProvider(spark)
# ⚠ REQUIRED: Set your Microsoft Sentinel workspace name before running this notebook
workspace_name = "<YOUR_WORKSPACE_NAME>" # ← Replace with your Sentinel workspace name



# =============================================================================
# Read DeviceNetworkEvents — single table provides BOTH DNS queries AND connections
#
# CRITICAL: For DnsConnectionInspected events, RemoteUrl is ALWAYS EMPTY.
# The actual queried domain lives in AdditionalFields.query and the
# resolved IPs are in AdditionalFields.answers (JSON array).
# RemoteIP for DNS events is just the DNS server (e.g., 168.63.129.16).
#
# For ConnectionSuccess events, RemoteUrl contains the domain and
# RemoteIP contains the actual destination IP.
# =============================================================================
net_raw = sentinel_provider.read_table('DeviceNetworkEvents', workspace_name)
net_raw = net_raw.df if hasattr(net_raw, 'df') else net_raw
net_base = (net_raw
 .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
)

# DNS query events — extract domain from AdditionalFields.query
# AdditionalFields also contains: rcode_name, answers (resolved IPs), qtype_name
# NOTE: DeviceId is included so Queried edges can match Device node key (DeviceId)
dns_df = (net_base
 .filter(col('ActionType') == 'DnsConnectionInspected')
 .withColumn("QueryName", get_json_object(col("AdditionalFields"), "$.query"))
 .withColumn("DnsResponseCode", get_json_object(col("AdditionalFields"), "$.rcode_name"))
 .withColumn("DnsAnswers", get_json_object(col("AdditionalFields"), "$.answers"))
 .where(col('QueryName').isNotNull() & (col('QueryName') != ''))
 .where(~col('QueryName').endswith('.arpa')) # Exclude reverse DNS lookups
 .select(
 col("DeviceId"),
 col("DeviceName"),
 col("QueryName"),
 col("DnsResponseCode"),
 col("DnsAnswers"),
 col("InitiatingProcessFileName"),
 col("TimeGenerated")
 )
).persist()

# Network connection events — public IP connections for Domain→ResolvedIP resolution mapping
net_df = (net_base
 .filter(
 (col('ActionType') == 'ConnectionSuccess') &
 (col('RemoteIPType') == 'Public') &
 (col('RemoteUrl').isNotNull()) & (col('RemoteUrl') != '')
 )
 .select(
 "DeviceName", "RemoteIP", "RemoteUrl", "RemoteIPType",
 "RemotePort", "Protocol", "TimeGenerated"
 )
).persist()

# =============================================================================
# Read DeviceInfo — endpoint device inventory
# Schema: DeviceId, DeviceName, OSPlatform, ExposureLevel, DeviceType, IsInternetFacing
# =============================================================================
device_raw = sentinel_provider.read_table('DeviceInfo', workspace_name)
device_raw = device_raw.df if hasattr(device_raw, 'df') else device_raw
device_df = (device_raw
 .filter(col('TimeGenerated') >= expr(f"current_timestamp() - INTERVAL {TIME_WINDOW_DAYS} DAYS"))
 .select(
 "DeviceId", "DeviceName", "OSPlatform", "ExposureLevel",
 "DeviceType", "IsInternetFacing", "PublicIP", "TimeGenerated"
 )
 .dropDuplicates(["DeviceId"])
).persist()

# =============================================================================
# Read ThreatIntelIndicators — STIX-format IOCs
#
# Migration context (Microsoft Tech Community, Aug 2025):
# Old table: ThreatIntelligenceIndicator — deprecated Aug 2025, retires May 2026
# New table: ThreatIntelIndicators — STIX schema, actively syncing
#
# Key schema differences:
# Old: IndicatorId, ThreatType, ConfidenceScore, NetworkIP, DomainName
# New: Id, ObservableKey, ObservableValue, Confidence, Tags, Data, SourceSystem
#
# ObservableKey encodes the IOC type:
# "domain-name:value" → ObservableValue = domain (e.g., "update.googleapis.com")
# "ipv4-addr:value" → ObservableValue = IP (e.g., "142.250.73.131")
# "file:hashes.'SHA-256'" → ObservableValue = hash (not used in this graph)
#
# We filter to domain + IP indicators only (relevant for DNS/network graph).
# =============================================================================
ti_raw = sentinel_provider.read_table('ThreatIntelIndicators', workspace_name)
ti_raw = ti_raw.df if hasattr(ti_raw, 'df') else ti_raw
ti_df = (ti_raw
 .filter(
 (col('IsActive') == True) &
 (col('IsDeleted') == False) &
 (col('ValidUntil') >= expr("current_timestamp()"))
 )
 .filter(
 col('ObservableKey').isin("domain-name:value", "ipv4-addr:value")
 )
 .select(
 "Id", "ObservableKey", "ObservableValue", "Confidence",
 "Tags", "Data", "SourceSystem", "ValidUntil", "TimeGenerated"
 )
).persist()

# Validation counts
print(f"📊 DeviceNetworkEvents (DNS queries, from AdditionalFields.query): {dns_df.count()} rows")
print(f"📊 DeviceNetworkEvents (public connections): {net_df.count()} rows")
print(f"📊 DeviceInfo: {device_df.count()} rows")
print(f"📊 ThreatIntelIndicators (active, domain + IP): {ti_df.count()} rows")

# Show sample TI indicators to verify new STIX schema
ti_df.select("Id", "ObservableKey", "ObservableValue", "Confidence", "SourceSystem").show(5, truncate=50)

## Node DataFrames

This is the core transformation step: we take the raw Sentinel tables and reshape them into **node DataFrames** (entities) and **edge DataFrames** (relationships) that the graph SDK expects.

### How Nodes Are Built
Each node type gets its own DataFrame with a `nodeType` column. The **key column** must uniquely identify each node:
- `Device` nodes -> keyed by `DeviceId` (a GUID like `a8f3k2x9-1234-5678-9abc-def012345678`)
- `Domain` nodes -> keyed by `QueryName` (the full domain string like `blob.core.windows.net`)
- `ResolvedIP` nodes -> keyed by `RemoteIP` (the IP address like `52.168.112.66`)
- `ThreatIndicator` nodes -> keyed by `Id` (STIX indicator ID, e.g., `TWljcm9zb2Z0...`)

### How Edges Are Built
Each edge connects a **source node** to a **target node** using their key columns. Edges are aggregated and enriched:

| Edge | How it's built | Aggregation |
|------|---------------|-------------|
| **Queried** | Group `dns_df` by (`DeviceId`, `QueryName`) | Count queries, track FirstSeen/LastSeen, capture TopProcess |
| **SubdomainOf** | Extract parent domain from `QueryName` using dot-splitting | Distinct pairs only |
| **ResolvesTo** | Group `net_df` by (`RemoteUrl`, `RemoteIP`) | Count connections, track FirstSeen |
| **MatchesTI (IP)** | Inner join `ResolvedIP` nodes with TI indicators where `ObservableKey == "ipv4-addr:value"`, on `RemoteIP == ObservableValue` | One edge per match |
| **MatchesTI (Domain)** | Inner join `Domain` nodes with TI indicators where `ObservableKey == "domain-name:value"`, on `QueryName == ObservableValue` | One edge per match |

### Parent Domain Extraction Logic
A key component is the **parent domain extraction** -- grouping subdomains under their registrable root. Naively taking the last 2 segments (e.g., `windows.net` from `blob.core.windows.net`) collapses 69+ unrelated Azure services into one bucket. We use **last-3-segments** for domains with 4+ parts:

| Domain | Naive (last-2) | This notebook (last-3 for 4+ parts) |
|--------|---------------|--------------------------------------|
| `wsus2eastprod.blob.core.windows.net` | `windows.net`  | `core.windows.net`  |
| `gsm1204297060eh.servicebus.windows.net` | `windows.net`  | `servicebus.windows.net`  |
| `md-ssd-xxx.z19.blob.storage.azure.net` | `azure.net`  | `storage.azure.net`  |
| `www.microsoft.com` (3 parts) | `microsoft.com`  | `microsoft.com`  |

Edge DataFrames are built in the same cell below, connecting nodes through relationship types.


In [ ]:
from pyspark.sql.functions import (
 col, lit, concat_ws, split, size, when,
 count as spark_count, min as spark_min, max as spark_max,
 first as spark_first, regexp_extract, expr, array_join,
 lag, avg as spark_avg, stddev as spark_stddev,
 countDistinct, hour, round as spark_round,
 unix_timestamp
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# =============================================================================
# PARENT DOMAIN UDF — Azure/cloud-aware multi-part TLD handling
# =============================================================================
# The naive "last 2 segments" approach collapses 69+ unrelated Azure domains
# into "windows.net". We use last-3-segments as default for 4+ part domains,
# with a known multi-part suffix list to correctly group Azure services.
#
# Evidence from live Sentinel data (14d, 20K+ DnsConnectionInspected events):
# - "windows.net" alone had 69 unique domains across blob, servicebus, etc.
# - "cloudapp.net" had 38 domains from internal.cloudapp.net
# - "azure.net" had 15 across storage, pki, attest services
#
# With this fix:
# wsus2eastprod.blob.core.windows.net → core.windows.net (not windows.net)
# gsm1204297060eh.servicebus.windows.net → servicebus.windows.net
# md-ssd-xxx.z19.blob.storage.azure.net → storage.azure.net

# Known Azure/cloud multi-part suffixes where last-2 is a public suffix,
# not a registrable domain. For these, we use last-3 segments.
MULTI_PART_TLDS = [
 "core.windows.net", "servicebus.windows.net",
 "internal.cloudapp.net", "storage.azure.net",
 "pki.azure.net", "attest.azure.net",
 "svc.cloud.microsoft", "dev.microsoft",
 "static.microsoft", "usercontent.microsoft",
 "azure-devices.net", "azurewebsites.net",
 "azureedge.net", "trafficmanager.net",
]

def extract_parent_domain(parts_col, size_col):
 """
 Extract the registrable parent domain from a split domain array.
 - 3-part domains (e.g., fs.microsoft.com): use last 2 → microsoft.com
 - 4+ part domains: use last 3 → services.visualstudio.com
 This properly separates blob.core.windows.net from servicebus.windows.net
 """
 last2 = concat_ws(".",
 parts_col[size_col - 2],
 parts_col[size_col - 1]
 )
 last3 = concat_ws(".",
 parts_col[size_col - 3],
 parts_col[size_col - 2],
 parts_col[size_col - 1]
 )
 # For 3-part domains: only last-2 is available as parent
 # For 4+ part domains: use last-3 (better Azure separation)
 return when(size_col == 3, last2).otherwise(last3)


# =============================================================================
# NODE DATAFRAMES (5 types)
# =============================================================================

# --- Device nodes ---
# From DeviceInfo — endpoint device inventory. DeviceId is the KEY used
# to join with Queried edges (DeviceNetworkEvents also has DeviceId natively).
device_nodes_df = (device_df
 .select(
 col("DeviceId"),
 col("DeviceName"),
 col("OSPlatform"),
 col("ExposureLevel"),
 col("DeviceType"),
 col("IsInternetFacing"),
 )
 .where(col("DeviceId").isNotNull())
 .withColumn("nodeType", lit("Device"))
)

# --- Domain nodes ---
# Every unique domain queried via MDE DNS inspection (DnsConnectionInspected)
domain_nodes_df = (dns_df
 .select(col("QueryName"))
 .distinct()
 .where(col("QueryName").isNotNull() & (col("QueryName") != ""))
 .withColumn("nodeType", lit("Domain"))
)

# --- ParentDomain nodes ---
# Smart parent domain extraction: last-2 for 3-part, last-3 for 4+ part domains
# This separates Azure services (core.windows.net vs servicebus.windows.net)
# instead of lumping them all under "windows.net"
parent_domain_df = (dns_df
 .select(col("QueryName"))
 .where(col("QueryName").isNotNull() & (col("QueryName") != ""))
 .withColumn("parts", split(col("QueryName"), "\\."))
 .withColumn("n", size(col("parts")))
 .where(col("n") >= 3) # Only subdomains (3+ parts)
 .withColumn("ParentDomain", extract_parent_domain(col("parts"), col("n")))
 .select("ParentDomain")
 .distinct()
 .withColumn("nodeType", lit("ParentDomain"))
)

# --- ResolvedIP nodes ---
# Public IPs that domains resolved to, from DeviceNetworkEvents ConnectionSuccess
resolved_ip_nodes_df = (net_df
 .select(
 col("RemoteIP"),
 col("RemoteIPType")
 )
 .distinct()
 .where(col("RemoteIP").isNotNull())
 .withColumn("nodeType", lit("ResolvedIP"))
)

# --- ThreatIndicator nodes ---
# Active TI indicators from ThreatIntelIndicators (STIX format)
# ObservableKey identifies IOC type ("domain-name:value", "ipv4-addr:value")
# ObservableValue is the actual IOC (domain string or IP address)
ti_nodes_df = (ti_df
 .select(
 col("Id"),
 col("ObservableKey"),
 col("ObservableValue"),
 col("Confidence"),
 col("SourceSystem"),
 col("ValidUntil")
 )
 .where(col("Id").isNotNull())
 .distinct()
 .withColumn("nodeType", lit("ThreatIndicator"))
)

# =============================================================================
# EDGE DATAFRAMES (5 types)
# =============================================================================

# --- Queried: Device → Domain (with beaconing metrics) ---
# Aggregate DNS queries per (DeviceId, QueryName) with count, first/last seen,
# top process, and BEACONING METRICS (interval regularity, 24/7 activity, etc.)
# that enable detection of C2 beacon patterns directly on the graph edge.
#
# Beaconing metrics:
# MeanIntervalSec — avg seconds between consecutive queries (beacon heartbeat)
# StddevIntervalSec — stddev of intervals (low = mechanical regularity)
# CoeffOfVariation — stddev/mean (< 0.3 = strong beacon signal)
# ActiveHoursRatio — fraction of 24h with queries (C2 ≈ 1.0, human ≈ 0.3)
# UniqueAnswerCount — distinct DNS answers (high = fast-flux C2)
# NxdomainRatio — fraction of NXDOMAIN responses (high = DGA pattern)
#
# IMPORTANT: Group by DeviceId (not DeviceName) because Device node key = DeviceId.
# DeviceName is kept via first() for display/debugging purposes.

# Step 1: Compute per-row inter-query interval using Window lag
_beacon_window = Window.partitionBy("DeviceId", "QueryName").orderBy("TimeGenerated")

_dns_with_intervals = (dns_df
 .withColumn("_prev_ts", lag("TimeGenerated", 1).over(_beacon_window))
 .withColumn("_interval_sec",
 when(col("_prev_ts").isNotNull(),
 (unix_timestamp(col("TimeGenerated")) - unix_timestamp(col("_prev_ts"))).cast("double")
 )
 )
 .withColumn("_hour", hour(col("TimeGenerated")))
 .withColumn("_is_nxdomain",
 when(col("DnsResponseCode").isin("NXDOMAIN", "NXDomain", "nxdomain"), lit(1)).otherwise(lit(0))
 )
)

# Step 2: Aggregate with both original + beaconing metrics
queried_edges_df = (_dns_with_intervals
 .groupBy("DeviceId", "QueryName")
 .agg(
 # --- Original metrics ---
 spark_count("*").alias("QueryCount"),
 spark_min("TimeGenerated").alias("FirstSeen"),
 spark_max("TimeGenerated").alias("LastSeen"),
 spark_first("InitiatingProcessFileName", ignorenulls=True).alias("TopProcess"),
 spark_first("DeviceName", ignorenulls=True).alias("DeviceName"),
 # --- Beaconing: interval regularity ---
 spark_round(spark_avg("_interval_sec"), 2).alias("MeanIntervalSec"),
 spark_round(spark_stddev("_interval_sec"), 2).alias("StddevIntervalSec"),
 # --- Beaconing: 24/7 activity ---
 countDistinct("_hour").alias("ActiveHoursCount"),
 # --- Beaconing: DNS answer diversity ---
 countDistinct("DnsAnswers").alias("UniqueAnswerCount"),
 # --- Beaconing: NXDOMAIN ratio ---
 F.sum("_is_nxdomain").alias("_nxdomain_count")
 )
 .where(col("DeviceId").isNotNull() & col("QueryName").isNotNull())
 # Derived beaconing metrics
 .withColumn("CoeffOfVariation",
 spark_round(
 when(col("MeanIntervalSec") > 0,
 col("StddevIntervalSec") / col("MeanIntervalSec")
 ),
 2)
 )
 .withColumn("ActiveHoursRatio",
 spark_round(col("ActiveHoursCount").cast("double") / lit(24.0), 2)
 )
 .withColumn("NxdomainRatio",
 spark_round(
 when(col("QueryCount") > 0,
 col("_nxdomain_count").cast("double") / col("QueryCount").cast("double")
 ),
 2)
 )
 .drop("_nxdomain_count", "ActiveHoursCount")
 .withColumn("edgeType", lit("Queried"))
 .withColumn("EdgeKey", concat_ws("_", col("DeviceId"), col("QueryName")))
)

# --- SubdomainOf: Domain → ParentDomain ---
# Link each domain to its parent using the same smart extraction logic
subdomain_edges_df = (dns_df
 .select(col("QueryName"))
 .where(col("QueryName").isNotNull() & (col("QueryName") != ""))
 .withColumn("parts", split(col("QueryName"), "\\."))
 .withColumn("n", size(col("parts")))
 .where(col("n") >= 3)
 .withColumn("ParentDomain", extract_parent_domain(col("parts"), col("n")))
 .select("QueryName", "ParentDomain")
 .distinct()
 .withColumn("edgeType", lit("SubdomainOf"))
 .withColumn("EdgeKey", concat_ws("_", col("QueryName"), col("ParentDomain")))
)

# --- ResolvesTo: Domain → ResolvedIP ---
# From DeviceNetworkEvents ConnectionSuccess: RemoteUrl (domain) → RemoteIP
resolves_to_edges_df = (net_df
 .groupBy("RemoteUrl", "RemoteIP")
 .agg(
 spark_count("*").alias("ConnectionCount"),
 spark_min("TimeGenerated").alias("FirstSeen")
 )
 .where(col("RemoteUrl").isNotNull() & col("RemoteIP").isNotNull())
 .withColumnRenamed("RemoteUrl", "QueryName") # Align with Domain node key
 .withColumn("edgeType", lit("ResolvesTo"))
 .withColumn("EdgeKey", concat_ws("_", col("QueryName"), col("RemoteIP")))
)

# --- MatchesTI: ResolvedIP → ThreatIndicator (IP match) ---
# Join resolved IPs with TI indicators where ObservableKey = "ipv4-addr:value"
# ObservableValue contains the IP address string
ti_ip_match_edges_df = (resolved_ip_nodes_df
 .join(ti_df.where(col("ObservableKey") == "ipv4-addr:value"),
 resolved_ip_nodes_df.RemoteIP == ti_df.ObservableValue, "inner")
 .select(
 col("RemoteIP"),
 col("Id"),
 lit("IP").alias("MatchType")
 )
 .withColumn("edgeType", lit("MatchesTI"))
 .withColumn("EdgeKey", concat_ws("_", col("RemoteIP"), col("Id")))
)

# --- MatchesTI: Domain → ThreatIndicator (Domain match) ---
# Join queried domains with TI indicators where ObservableKey = "domain-name:value"
# ObservableValue contains the domain name string
ti_domain_match_edges_df = (domain_nodes_df
 .join(ti_df.where(col("ObservableKey") == "domain-name:value"),
 domain_nodes_df.QueryName == ti_df.ObservableValue, "inner")
 .select(
 col("QueryName"),
 col("Id"),
 lit("Domain").alias("MatchType")
 )
 .withColumn("edgeType", lit("MatchesTI"))
 .withColumn("EdgeKey", concat_ws("_", col("QueryName"), col("Id")))
)


# =============================================================================
# Validation counts
# =============================================================================
print(f"📊 Nodes - Devices: {device_nodes_df.count()}, Domains: {domain_nodes_df.count()}, ParentDomains: {parent_domain_df.count()}")
print(f"📊 Nodes - ResolvedIPs: {resolved_ip_nodes_df.count()}, ThreatIndicators: {ti_nodes_df.count()}")
print(f"📊 Edges - Queried: {queried_edges_df.count()}, SubdomainOf: {subdomain_edges_df.count()}")
print(f"📊 Edges - ResolvesTo: {resolves_to_edges_df.count()}, MatchesTI (IP): {ti_ip_match_edges_df.count()}, MatchesTI (Domain): {ti_domain_match_edges_df.count()}")


## Graph Assembly

This cell wires everything together using the `GraphSpecBuilder` fluent API. For each node and edge type, we specify:

- **`from_dataframe(df)`** -- the PySpark DataFrame containing the data
- **`with_columns(..., key=, display=)`** -- which columns to include as properties, which column is the **unique key** (used for joining edges), and which column is the **display label** (shown in the visual graph)
- **`source(id_column=, node_type=)`** / **`target(id_column=, node_type=)`** -- for edges: which column in the edge DataFrame maps to which node type's key

**Critical rule**: The `id_column` in `.source()` or `.target()` must contain the **same values** as the target node's `key` column. For example, the `Queried` edge uses `.source(id_column="DeviceId")` because the `Device` node has `key="DeviceId"` -- both are GUIDs. The `MatchesTI` edge uses `.target(id_column="Id")` because the `ThreatIndicator` node has `key="Id"`.

After calling `.done()`, the builder is ready to materialize the graph. The next cell calls `Graph.build(spec)` to prepare and publish the graph.

In [ ]:

dns_beacon_graph = (GraphSpecBuilder.start()

 # ===================== NODES (5 types) =====================

 .add_node("Device")
 .from_dataframe(device_nodes_df)
 .with_columns("DeviceId", "DeviceName", "OSPlatform", "ExposureLevel",
 "DeviceType", "IsInternetFacing", "nodeType",
 key="DeviceId", display="DeviceName")

 .add_node("Domain")
 .from_dataframe(domain_nodes_df)
 .with_columns("QueryName", "nodeType",
 key="QueryName", display="QueryName")

 .add_node("ParentDomain")
 .from_dataframe(parent_domain_df)
 .with_columns("ParentDomain", "nodeType",
 key="ParentDomain", display="ParentDomain")

 .add_node("ResolvedIP")
 .from_dataframe(resolved_ip_nodes_df)
 .with_columns("RemoteIP", "RemoteIPType", "nodeType",
 key="RemoteIP", display="RemoteIP")

 .add_node("ThreatIndicator")
 .from_dataframe(ti_nodes_df)
 .with_columns("Id", "ObservableKey", "ObservableValue", "Confidence",
 "SourceSystem", "ValidUntil", "nodeType",
 key="Id", display="ObservableValue")

 # ===================== EDGES (5 types) =====================

 # Queried: Device → Domain (with beaconing metrics)
 .add_edge("Queried")
 .from_dataframe(queried_edges_df)
 .source(id_column="DeviceId", node_type="Device")
 .target(id_column="QueryName", node_type="Domain")
 .with_columns("edgeType", "QueryCount", "FirstSeen", "LastSeen",
 "TopProcess", "DeviceName",
 "MeanIntervalSec", "StddevIntervalSec", "CoeffOfVariation",
 "ActiveHoursRatio", "UniqueAnswerCount", "NxdomainRatio",
 "EdgeKey",
 key="EdgeKey", display="edgeType")

 .add_edge("SubdomainOf")
 .from_dataframe(subdomain_edges_df)
 .source(id_column="QueryName", node_type="Domain")
 .target(id_column="ParentDomain", node_type="ParentDomain")
 .with_columns("edgeType", "EdgeKey",
 key="EdgeKey", display="edgeType")

 .add_edge("ResolvesTo")
 .from_dataframe(resolves_to_edges_df)
 .source(id_column="QueryName", node_type="Domain")
 .target(id_column="RemoteIP", node_type="ResolvedIP")
 .with_columns("edgeType", "ConnectionCount", "FirstSeen", "EdgeKey",
 key="EdgeKey", display="edgeType")

 .add_edge("MatchesTI")
 .from_dataframe(ti_ip_match_edges_df)
 .source(id_column="RemoteIP", node_type="ResolvedIP")
 .target(id_column="Id", node_type="ThreatIndicator")
 .with_columns("edgeType", "MatchType", "EdgeKey",
 key="EdgeKey", display="edgeType")

 .add_edge("MatchesTI")
 .from_dataframe(ti_domain_match_edges_df)
 .source(id_column="QueryName", node_type="Domain")
 .target(id_column="Id", node_type="ThreatIndicator")
 .with_columns("edgeType", "MatchType", "EdgeKey",
 key="EdgeKey", display="edgeType")

).done()



In [ ]:
# Check the schema of the graph spec to ensure it is correct
dns_beacon_graph.show_schema()


## Build & Publish

Build the graph from the spec. This validates the schema, prepares the data, and publishes the graph for querying.


In [ ]:
# Build the graph from the spec - this will validate the spec and prepare it for querying
# Alternative: use Graph.prepare() to prepare the graph nodes and edges in the lake
# and then use Graph.publish() to create the graph. You would typically call prepare()
# and publish() separately to understand the cost of Graph API calls triggered by Graph.publish()
# see https://learn.microsoft.com/azure/sentinel/billing?tabs=simplified%2Ccommitment-tiers
dns_graph = Graph.build(dns_beacon_graph)
